# Notebook 08b — Benchmark Sparse Retrieval và Fusion (BM25, TF-IDF, RRF, Weighted)

Notebook này thực hiện đánh giá thực nghiệm có đối chứng (controlled benchmark) cho các phương pháp truy xuất Sparse (BM25, TF-IDF) và các kỹ thuật kết hợp đa tầng (RRF, Min-Max Weighted Fusion, Dense-BM25 Rescoring) trên tập 572 chunks ẩm thực và 45 câu hỏi đánh giá chuẩn **Golden Dataset V3**.

> [!IMPORTANT]
> **Nguyên tắc Cách ly Production & Không Phải Cutover**:
> - Benchmark chạy hoàn toàn trên các collection và cấu hình thử nghiệm cô lập.
> - Collection production đang hoạt động (`hue_foods_e5_small_384`) được snapshot và bảo vệ nghiêm ngặt, không bị chỉnh sửa (read-only).
> - Kết quả benchmark đóng vai trò cung cấp bằng chứng thực nghiệm để đề cử ứng viên (finalist) cho giai đoạn tiếp theo; notebook này không thực hiện thay đổi cấu hình production runtime.

## 1. Thiết lập Môi trường và Đường dẫn

In [20]:
import sys
from pathlib import Path

# Thêm backend vào sys.path nếu đang chạy từ thư mục notebooks/
backend_path = Path.cwd().parent / "backend" if Path.cwd().name == "notebooks" else Path.cwd() / "backend"
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

print(f"Backend path: {backend_path}")

Backend path: /home/minhhieu/hue_rag/backend


### 1.1 Thông tin Môi trường Thực thi

In [21]:
from evaluation.sparse_benchmark import environment_table
environment_table()

Property,Value
str,str
"""OS / Kernel""","""Linux 6.18.33.2-microsoft-stan…"
"""Python Version""","""3.13.15"""
"""Qdrant Client Version""","""1.19.0"""
"""Underthesea Version""","""9.5.0"""


## 2. Nạp Dữ liệu Đầu vào và Kiểm tra Điều kiện Tiên quyết 08a

Xác thực tính toàn vẹn của 572 chunks, 45 Golden V3 cases và 3 collection dense vector từ Notebook 08a.

In [22]:
from evaluation.sparse_benchmark import (
    load_sparse_benchmark_inputs,
    validate_08a_prerequisites,
    snapshot_active_collection,
    canonical_inputs_table,
)

inputs = load_sparse_benchmark_inputs()
active_snapshot = snapshot_active_collection(inputs)
print(f"Active production collection: {active_snapshot.get('collection_name')} ({active_snapshot.get('points_count')} points)")
canonical_inputs_table(inputs)

Active production collection: hue_foods_e5_small_384 (572 points)


Item,Count / Value
str,str
"""Total Canonical Chunks""","""572"""
"""Total Golden V3 Cases""","""45"""
"""Corpus Fingerprint (12 chars)""","""5297c8604335"""
"""Golden V3 Fingerprint (12 char…","""8506eac3f567"""
"""Chunker Fingerprint (12 chars)""","""d5e61db34b76"""
…,…
"""Category: holistic""","""3"""
"""Category: numerical""","""2"""
"""Category: relationship""","""14"""


In [23]:
from evaluation.sparse_benchmark import dense_prerequisite_table

prereqs = validate_08a_prerequisites(inputs)
print(f"Đã xác thực thành công {len(prereqs)} cấu hình dense 08a prerequisites.")
dense_prerequisite_table(prereqs)

Đã xác thực thành công 3 cấu hình dense 08a prerequisites.


Dense Setting,Model ID,Dimension,Collection Name,Point Count,Recall@5,nDCG@5,MRR@5,p95 Latency (ms)
str,str,i64,str,i64,str,str,str,str
"""e5-small-384""","""intfloat/multilingual-e5-small""",384,"""hue_foods_08a_e5_small_384""",572,"""0.8185""","""0.7425""","""0.7748""","""46.5"""
"""huydang-dek21-embedding-768""","""CODE4LIFEOFFICIAL/huydang-dek2…",768,"""hue_foods_08a_huydang_dek21_76…",572,"""0.8370""","""0.7164""","""0.7211""","""80.3"""
"""e5-base-768""","""intfloat/multilingual-e5-base""",768,"""hue_foods_08a_e5_base_768""",572,"""0.8407""","""0.7061""","""0.6985""","""211.0"""


## 3. Phần A — BM25 và Tokenizer Calibration

Thực hiện hiệu chuẩn siêu tham số $k_1, b$ cho BM25 và so sánh hiệu quả giữa bộ tách từ `unicode_word` (chuẩn Unicode regex) và `underthesea_word` (tách từ ghép tiếng Việt).

In [24]:
from evaluation.sparse_benchmark import bm25_parameter_table
bm25_parameter_table()

Order,Setting Key,k1,b
i64,str,f64,f64
1,"""baseline""",1.5,0.75
2,"""k1_low""",1.2,0.75
3,"""k1_high""",1.8,0.75
4,"""b_low""",1.5,0.5
5,"""b_high""",1.5,1.0


In [25]:
import json
from evaluation.sparse_benchmark import (
    DEFAULT_RESULTS_DIR,
    MANIFEST_FILENAME,
    ExperimentManifest,
    load_checkpoint,
    load_or_run_calibration,
)

manifest_path = DEFAULT_RESULTS_DIR / MANIFEST_FILENAME
checkpoint_initial = load_checkpoint(ExperimentManifest.from_dict(json.loads(manifest_path.read_text(encoding="utf-8")))) if manifest_path.exists() else None
selected_lexical = load_or_run_calibration(inputs, expected_active_snapshot=active_snapshot, checkpoint=checkpoint_initial)
print(f"Selected BM25 parameters: key={selected_lexical.bm25_setting_key} (k1={selected_lexical.k1}, b={selected_lexical.b})")
print(f"Reason: {selected_lexical.parameter_selection_reason}")
print(f"Selected Tokenizer: key={selected_lexical.tokenizer_key}")
print(f"Reason: {selected_lexical.tokenizer_selection_reason}")

Selected BM25 parameters: key=baseline (k1=1.5, b=0.75)
Reason: Baseline optimal among valid candidates
Selected Tokenizer: key=unicode_word
Reason: Unicode word tokenizer retained (simplicity preference)


In [26]:
from evaluation.sparse_benchmark import calibration_table
manifest_path = DEFAULT_RESULTS_DIR / MANIFEST_FILENAME
if manifest_path.exists():
    checkpoint_temp = load_checkpoint(ExperimentManifest.from_dict(json.loads(manifest_path.read_text(encoding="utf-8"))))
    display(calibration_table(checkpoint_temp))
else:
    print("Chưa có file manifest.")

experiment_version,calibration_stage,setting_key,category,tokenizer_key,k1,b,status,error,case_count,hit_case_count,recall_at_30,mrr_at_5,ndcg_at_5,successful_repetitions,ranking_stable,build_ms,warm_total_p50_ms,warm_total_p95_ms,observed_peak_rss_mb,delta_recall_at_30,delta_mrr_at_5,delta_ndcg_at_5,category_guardrail_pass,all_category_guardrails_pass,selected,selection_reason
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""phase8-08b-v1""","""parameter""","""b_high""","""overall""","""unicode_word""","""1.5""","""1.0""","""completed""","""""","""45""","""44""","""0.951851851851852""","""0.6551851851851852""","""0.6442220666254881""","""3""","""True""","""71.53990200004046""","""1.3195219999033725""","""1.8899269000485217""","""3.294276237487793""","""0.0""","""-0.00851851851851848""","""-0.0035714596386143826""","""True""","""True""","""false""",""""""
"""phase8-08b-v1""","""parameter""","""b_low""","""overall""","""unicode_word""","""1.5""","""0.5""","""completed""","""""","""45""","""44""","""0.937037037037037""","""0.6755555555555556""","""0.6580598485725545""","""3""","""True""","""68.88279999975566""","""1.5028540001367219""","""2.2720914997989894""","""3.294283866882324""","""-0.014814814814814947""","""0.011851851851851891""","""0.010266322308452036""","""False""","""False""","""false""",""""""
"""phase8-08b-v1""","""parameter""","""baseline""","""overall""","""unicode_word""","""1.5""","""0.75""","""completed""","""""","""45""","""44""","""0.951851851851852""","""0.6637037037037037""","""0.6477935262641025""","""3""","""True""","""76.18226799968397""","""1.4855280001029314""","""2.093277199946897""","""3.296938896179199""","""0.0""","""0.0""","""0.0""","""True""","""True""","""true""","""Baseline optimal among valid c…"
"""phase8-08b-v1""","""parameter""","""k1_high""","""overall""","""unicode_word""","""1.8""","""0.75""","""completed""","""""","""45""","""44""","""0.951851851851852""","""0.6637037037037037""","""0.6483906464819534""","""3""","""True""","""69.12676000001738""","""1.8000430000029155""","""2.888639000002513""","""3.2942914962768555""","""0.0""","""0.0""","""0.000597120217850966""","""False""","""False""","""false""",""""""
"""phase8-08b-v1""","""parameter""","""k1_low""","""overall""","""unicode_word""","""1.2""","""0.75""","""completed""","""""","""45""","""44""","""0.951851851851852""","""0.6788888888888889""","""0.659581479328605""","""3""","""True""","""73.3243049999146""","""1.5861990000303194""","""2.5681085997803166""","""3.2942991256713867""","""0.0""","""0.01518518518518519""","""0.011787953064502532""","""False""","""False""","""false""",""""""
"""phase8-08b-v1""","""tokenizer""","""underthesea_word""","""overall""","""underthesea_word""","""1.5""","""0.75""","""completed""","""""","""45""","""44""","""0.9148148148148147""","""0.5292592592592592""","""0.5337690005156664""","""3""","""True""","""4081.145002000085""","""1.8542570001045533""","""3.3656082001925824""","""2.8719844818115234""","""-0.0370370370370372""","""-0.13444444444444448""","""-0.11402452574843602""","""False""","""False""","""false""",""""""
"""phase8-08b-v1""","""tokenizer""","""unicode_word""","""overall""","""unicode_word""","""1.5""","""0.75""","""completed""","""""","""45""","""44""","""0.951851851851852""","""0.6637037037037037""","""0.6477935262641025""","""3""","""True""","""79.11188200023389""","""1.631959999940591""","""2.452153400054158""","""3.294314384460449""","""0.0""","""0.0""","""0.0""","""True""","""True""","""true""","""Unicode word tokenizer retaine…"


## 4. Phần B — TF-IDF và Catalog 20 Cấu hình Truy xuất

Khởi tạo hoặc kiểm tra collection sparse vector TF-IDF độc lập trên Qdrant với công thức Log-TF Smoothed-IDF L2 normalization và thiết lập danh mục 20 cấu hình truy xuất.

In [27]:
from evaluation.sparse_benchmark import build_or_validate_tfidf, retrieval_settings_table

tfidf_state = build_or_validate_tfidf(
    inputs.client,
    inputs.chunks,
    selected_lexical.tokenizer,
    selected_lexical.tokenizer_key,
    expected_active_snapshot=active_snapshot,
)
print(f"TF-IDF sparse collection sẵn sàng: {tfidf_state.collection_name} (vocab size: {tfidf_state.encoder.vocab_size})")
retrieval_settings_table()

TF-IDF sparse collection sẵn sàng: hue_rag_phase8_08b_tfidf_v1_unicode_word_5297c8604335 (vocab size: 2093)


Order,Setting Key,Setting Label,Path,Dense Setting,Sparse Family,Fusion Method
i64,str,str,str,str,str,str
1,"""dense__e5-small-384""","""Dense: E5-small 384D""","""dense_only""","""e5-small-384""","""-""","""-"""
2,"""dense__huydang-dek21-embedding…","""Dense: Huydang DEk21 768D""","""dense_only""","""huydang-dek21-embedding-768""","""-""","""-"""
3,"""dense__e5-base-768""","""Dense: E5-base 768D""","""dense_only""","""e5-base-768""","""-""","""-"""
4,"""bm25-only""","""Sparse: BM25 Only""","""sparse_only""","""-""","""bm25""","""-"""
5,"""dense-bm25-rescore__e5-small-3…","""Rescore: E5-small -> BM25""","""dense_bm25_rescore""","""e5-small-384""","""bm25""","""weighted"""
…,…,…,…,…,…,…
16,"""hybrid-tfidf-weighted__e5-smal…","""Hybrid: E5-small + TF-IDF (Wei…","""hybrid_dense_sparse""","""e5-small-384""","""tfidf""","""weighted"""
17,"""hybrid-tfidf-rrf__huydang-dek2…","""Hybrid: Huydang DEk21 + TF-IDF…","""hybrid_dense_sparse""","""huydang-dek21-embedding-768""","""tfidf""","""rrf"""
18,"""hybrid-tfidf-weighted__huydang…","""Hybrid: Huydang DEk21 + TF-IDF…","""hybrid_dense_sparse""","""huydang-dek21-embedding-768""","""tfidf""","""weighted"""


## 5. Chạy Thực nghiệm Tuần tự và Lưu Checkpoint

Thực thi tuần tự 20 cấu hình với 3 lần lặp (repetitions) cho mỗi câu hỏi để đo độ trễ và độ ổn định thứ hạng. Kết quả được lưu checkpoint nguyên tử (atomic) sau mỗi cấu hình.

In [28]:
import os
from evaluation.sparse_benchmark import (
    requested_setting_keys_from_env,
    run_retrieval_batch,
)

requested_keys = requested_setting_keys_from_env(os.environ.get("HUE_RAG_08B_SETTING_KEYS"))
print(f"Danh sách cấu hình cần chạy ({len(requested_keys)} settings): {requested_keys}")

for result in run_retrieval_batch(
    inputs,
    selected_lexical,
    tfidf_state,
    requested_setting_keys=requested_keys,
    expected_active_snapshot=active_snapshot,
):
    if result.status == "completed":
        r5 = float(result.summary["recall_at_5"]) if result.summary.get("recall_at_5") else 0.0
        ndcg5 = float(result.summary["ndcg_at_5"]) if result.summary.get("ndcg_at_5") else 0.0
        p95 = float(result.summary["warm_total_p95_ms"]) if result.summary.get("warm_total_p95_ms") else 0.0
        print(f"[✓] Setting #{result.setting.order:02d}: {result.setting.setting_key:<45} | Recall@5: {r5:.4f} | nDCG@5: {ndcg5:.4f} | p95: {p95:.1f}ms")
    else:
        print(f"[✗] Setting #{result.setting.order:02d}: {result.setting.setting_key:<45} | FAILED: {result.error}")


Danh sách cấu hình cần chạy (20 settings): ('dense__e5-small-384', 'dense__huydang-dek21-embedding-768', 'dense__e5-base-768', 'bm25-only', 'dense-bm25-rescore__e5-small-384', 'dense-bm25-rescore__huydang-dek21-embedding-768', 'dense-bm25-rescore__e5-base-768', 'hybrid-bm25-rrf__e5-small-384', 'hybrid-bm25-weighted__e5-small-384', 'hybrid-bm25-rrf__huydang-dek21-embedding-768', 'hybrid-bm25-weighted__huydang-dek21-embedding-768', 'hybrid-bm25-rrf__e5-base-768', 'hybrid-bm25-weighted__e5-base-768', 'tfidf-only', 'hybrid-tfidf-rrf__e5-small-384', 'hybrid-tfidf-weighted__e5-small-384', 'hybrid-tfidf-rrf__huydang-dek21-embedding-768', 'hybrid-tfidf-weighted__huydang-dek21-embedding-768', 'hybrid-tfidf-rrf__e5-base-768', 'hybrid-tfidf-weighted__e5-base-768')
[✓] Setting #01: dense__e5-small-384                           | Recall@5: 0.8185 | nDCG@5: 0.7425 | p95: 77.5ms
[✓] Setting #02: dense__huydang-dek21-embedding-768            | Recall@5: 0.8370 | nDCG@5: 0.7164 | p95: 116.1ms
[✓] Setti

## 6. Đối soát Toàn vẹn (Reconciliation) và Phân tích Đánh giá

Sau khi toàn bộ 20 cấu hình hoàn thành, đối soát toàn bộ 200 dòng CSV, 900 bản ghi case, tính toán khoảng tin cậy Bootstrap 95%, kiểm tra các cổng chất lượng và lựa chọn Finalist cho từng họ sparse.

In [29]:
from evaluation.sparse_benchmark import (
    load_checkpoint_for_inputs,
    reconcile_sparse_benchmark,
    artifact_reconciliation_table,
)

checkpoint_final = load_checkpoint_for_inputs(inputs, selected_lexical, tfidf_state)
reconciliation = reconcile_sparse_benchmark(
    checkpoint_final,
    inputs=inputs,
    expected_active_snapshot=active_snapshot,
    client=inputs.client,
    tfidf_state=tfidf_state,
)
print(f"Trạng thái đối soát toàn vẹn: {reconciliation.complete}")
print(f"Tóm tắt: {reconciliation.summary}")
print(f"BM25 Finalist: {reconciliation.bm25_finalist}")
print(f"TF-IDF Finalist: {reconciliation.tfidf_finalist}")
artifact_reconciliation_table(checkpoint_final)

Trạng thái đối soát toàn vẹn: True
Tóm tắt: {'bm25_parameter_settings_completed': 5, 'tokenizer_settings_completed': 2, 'main_settings_completed': 20, 'total_calibration_rows': 70, 'total_result_rows': 200, 'total_case_records': 900, 'reconciliation_complete': True, 'bm25_finalist': None, 'tfidf_finalist': None}
BM25 Finalist: None
TF-IDF Finalist: None


Artifact,Status,Count / Status
str,str,str
"""Manifest JSON""","""Valid""",null
"""Calibration CSV Rows""",null,"""70"""
"""Results CSV Rows""",null,"""200"""
"""Case Records JSONL""",null,"""900"""
"""Completed Settings""",null,"""20 / 20"""


In [30]:
from evaluation.sparse_benchmark import quality_table
quality_table(checkpoint_final)

experiment_version,setting_order,setting_key,setting_label,category,path,dense_setting_key,sparse_family,fusion_method,status,error,case_count,hit_case_count,successful_repetitions,ranking_stable,dense_recall_at_30,sparse_recall_at_30,candidate_union_recall,fusion_recall_at_10,recall_at_5,mrr_at_5,ndcg_at_5,dense_query_p50_ms,dense_query_p95_ms,sparse_query_p50_ms,sparse_query_p95_ms,fusion_p50_ms,fusion_p95_ms,warm_total_p50_ms,warm_total_p95_ms,build_ms,observed_peak_rss_mb,delta_fusion_recall_at_10,delta_recall_at_5,delta_mrr_at_5,delta_ndcg_at_5,recall_ci_lower,recall_ci_upper,mrr_ci_lower,mrr_ci_upper,ndcg_ci_lower,ndcg_ci_upper,category_guardrail_pass,all_category_guardrails_pass,fusion_recall_gate,final_recall_gate,latency_gate,complete_gate,finalist_eligible,finalist_selected
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""phase8-08b-v1""","""1""","""dense__e5-small-384""","""Dense: E5-small 384D""","""overall""","""dense_only""","""e5-small-384""","""""","""""","""completed""","""""","""45""","""41""","""3""","""True""","""0.9925925925925926""","""""","""0.9925925925925926""","""0.9185185185185186""","""0.8185185185185185""","""0.7748148148148148""","""0.7425376173235151""","""62.879506000172114""","""77.43126359928283""","""0.0""","""0.0""","""0.03534700044838246""","""0.04435720002220478""","""62.913195999499294""","""77.45945979877433""","""""","""186.32050704956055""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""True""","""True""","""True""","""True""","""True""","""True""","""True""","""false"""
"""phase8-08b-v1""","""2""","""dense__huydang-dek21-embedding…","""Dense: Huydang DEk21 768D""","""overall""","""dense_only""","""huydang-dek21-embedding-768""","""""","""""","""completed""","""""","""45""","""40""","""3""","""True""","""0.9555555555555556""","""""","""0.9555555555555556""","""0.9074074074074074""","""0.837037037037037""","""0.7211111111111111""","""0.7163617882034018""","""104.17551600039587""","""116.02404719997139""","""0.0""","""0.0""","""0.027303000024403445""","""0.03452119963185396""","""104.20106700075848""","""116.05054140054563""","""""","""37.70852565765381""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""True""","""True""","""True""","""True""","""True""","""True""","""True""","""false"""
"""phase8-08b-v1""","""3""","""dense__e5-base-768""","""Dense: E5-base 768D""","""overall""","""dense_only""","""e5-base-768""","""""","""""","""completed""","""""","""45""","""42""","""3""","""True""","""0.9925925925925926""","""""","""0.9925925925925926""","""0.9407407407407408""","""0.8407407407407408""","""0.6985185185185185""","""0.7060878464719689""","""114.8599269999977""","""130.42096459976165""","""0.0""","""0.0""","""0.028824999390053563""","""0.038154999128892086""","""114.88474500038137""","""130.46193299996958""","""""","""183.9367036819458""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""0.0""","""True""","""True""","""True""","""True""","""True""","""True""","""True""","""false"""
"""phase8-08b-v1""","""4""","""bm25-only""","""Sparse: BM25 Only""","""overall""","""sparse_only""","""""","""bm25""","""""","""completed""","""""","""45""","""39""","""3""","""True""","""""","""0.951851851851852""","""0.951851851851852""","""0.8703703703703703""","""0.7888888888888889""","""0.6637037037037037""","""0.6477935262641025""","""0.0""","""0.0""","""9.793199000341701""","""14.587938799377296""","""0.015666999388486147""","""0.030442199567914923""","""9.807289001400932""","""14.607764198444782""","""""","""4.975237846374512""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""True""","""""","""""","""""","""True""","""""","""false"""
"""phase8-08b-v1""","""5""","""den

In [31]:
from evaluation.sparse_benchmark import stage_recall_table
stage_recall_table(checkpoint_final)

Setting Key,Dense Recall@30,Sparse Recall@30,Candidate Union Recall,Fusion Recall@10,Final Recall@5
str,str,str,str,str,str
"""dense__e5-small-384""","""0.9925925925925926""","""""","""0.9925925925925926""","""0.9185185185185186""","""0.8185185185185185"""
"""dense__huydang-dek21-embedding…","""0.9555555555555556""","""""","""0.9555555555555556""","""0.9074074074074074""","""0.837037037037037"""
"""dense__e5-base-768""","""0.9925925925925926""","""""","""0.9925925925925926""","""0.9407407407407408""","""0.8407407407407408"""
"""bm25-only""","""""","""0.951851851851852""","""0.951851851851852""","""0.8703703703703703""","""0.7888888888888889"""
"""dense-bm25-rescore__e5-small-3…","""0.9925925925925926""","""0.9925925925925926""","""0.9925925925925926""","""0.9333333333333333""","""0.862962962962963"""
…,…,…,…,…,…
"""hybrid-tfidf-weighted__e5-smal…","""0.9925925925925926""","""0.9481481481481481""","""1.0""","""0.9074074074074074""","""0.8407407407407408"""
"""hybrid-tfidf-rrf__huydang-dek2…","""0.9555555555555556""","""0.9481481481481481""","""0.9888888888888889""","""0.9296296296296297""","""0.8777777777777778"""
"""hybrid-tfidf-weighted__huydang…","""0.9555555555555556""","""0.9481481481481481""","""0.9888888888888889""","""0.9222222222222223""","""0.9111111111111111"""


In [32]:
from evaluation.sparse_benchmark import category_guardrail_table
category_guardrail_table(checkpoint_final)

Setting Key,Category,Candidate Recall@5,Candidate nDCG@5,Control nDCG@5,Δ nDCG@5,Guardrail Pass
str,str,str,str,str,str,str
"""bm25-only""","""comparative""","""0.7500""","""0.6097""","""-""","""-""",""""""
"""bm25-only""","""direct_fact""","""1.0000""","""0.7731""","""-""","""-""",""""""
"""bm25-only""","""food_knowledge""","""0.7857""","""0.6429""","""-""","""-""",""""""
"""bm25-only""","""guide_planning""","""0.5000""","""0.3155""","""-""","""-""",""""""
"""bm25-only""","""holistic""","""0.6667""","""0.3338""","""-""","""-""",""""""
…,…,…,…,…,…,…
"""hybrid-tfidf-weighted__e5-base…","""holistic""","""1.0000""","""0.5441""","""0.5850""","""-0.0409""","""True"""
"""hybrid-tfidf-weighted__e5-base…","""numerical""","""1.0000""","""1.0000""","""1.0000""","""+0.0000""","""True"""
"""hybrid-tfidf-weighted__e5-base…","""relationship""","""0.9286""","""0.8118""","""0.8148""","""-0.0030""","""True"""


In [33]:
from evaluation.sparse_benchmark import latency_resource_table
latency_resource_table(checkpoint_final)

Setting Key,Dense Query p95 (ms),Sparse Query p95 (ms),Fusion p95 (ms),Warm Total p50 (ms),Warm Total p95 (ms),Peak RSS (MB)
str,str,str,str,str,str,str
"""dense__e5-small-384""","""77.43126359928283""","""0.0""","""0.04435720002220478""","""62.913195999499294""","""77.45945979877433""","""186.32050704956055"""
"""dense__huydang-dek21-embedding…","""116.02404719997139""","""0.0""","""0.03452119963185396""","""104.20106700075848""","""116.05054140054563""","""37.70852565765381"""
"""dense__e5-base-768""","""130.42096459976165""","""0.0""","""0.038154999128892086""","""114.88474500038137""","""130.46193299996958""","""183.9367036819458"""
"""bm25-only""","""0.0""","""14.587938799377296""","""0.030442199567914923""","""9.807289001400932""","""14.607764198444782""","""4.975237846374512"""
"""dense-bm25-rescore__e5-small-3…","""71.83324000070569""","""18.41610919982486""","""11.823471199022604""","""85.42912899974908""","""105.62598620126663""","""187.42813205718994"""
…,…,…,…,…,…,…
"""hybrid-tfidf-weighted__e5-smal…","""71.1456315995747""","""10.80237480055075""","""1.4155503999063508""","""70.6835319979291""","""82.30794180090015""","""184.17957019805908"""
"""hybrid-tfidf-rrf__huydang-dek2…","""118.56101339944871""","""10.624704600922996""","""0.88597419889993""","""115.9148959977756""","""128.1462503986404""","""40.67112159729004"""
"""hybrid-tfidf-weighted__huydang…","""115.41963040035625""","""10.110643400548724""","""1.1916302002646262""","""116.5432720008539""","""125.04647700043279""","""40.73212146759033"""


In [34]:
from evaluation.sparse_benchmark import case_disagreement_table
case_disagreement_table(checkpoint_final)

Case ID,Category,Dense R@5,Sparse R@5,Hybrid R@5,Dense nDCG@5,Hybrid nDCG@5,Δ nDCG@5,Disagreement Type
str,str,str,str,str,str,str,str,str
"""foods-v3-0003""","""direct_fact""","""1.0000""","""1.0000""","""1.0000""","""0.6309""","""1.0000""","""+0.3691""","""Hybrid improved nDCG"""
"""foods-v3-0004""","""comparative""","""1.0000""","""1.0000""","""1.0000""","""0.9197""","""0.8772""","""-0.0425""","""Hybrid regressed nDCG"""
"""foods-v3-0005""","""comparative""","""0.5000""","""1.0000""","""1.0000""","""0.6131""","""0.8503""","""+0.2372""","""Hybrid improved Recall; Hybrid…"
"""foods-v3-0008""","""relationship""","""1.0000""","""0.0000""","""1.0000""","""0.8772""","""0.6509""","""-0.2263""","""Hybrid regressed nDCG; Dense b…"
"""foods-v3-0009""","""relationship""","""1.0000""","""1.0000""","""1.0000""","""0.5000""","""0.4307""","""-0.0693""","""Hybrid regressed nDCG"""
…,…,…,…,…,…,…,…,…
"""foods-v3-0039""","""spanning""","""0.1667""","""0.5000""","""0.3333""","""0.3392""","""0.4852""","""+0.1461""","""Hybrid improved Recall; Hybrid…"
"""foods-v3-0041""","""spanning""","""0.0000""","""0.5000""","""0.1667""","""0.0000""","""0.1312""","""+0.1312""","""Hybrid improved Recall; Hybrid…"
"""foods-v3-0042""","""holistic""","""1.0000""","""0.0000""","""1.0000""","""0.6309""","""0.6309""","""+0.0000""","""Dense beat Sparse"""


In [35]:
from evaluation.sparse_benchmark import bootstrap_finalist_table
bootstrap_finalist_table(checkpoint_final)

Setting Key,Path,Recall@5,Δ Recall@5,Recall 95% CI,nDCG@5,Δ nDCG@5,nDCG 95% CI,Guardrails Pass,Eligible,Selected Finalist
str,str,str,str,str,str,str,str,str,str,str
"""dense__e5-small-384""","""dense_only""","""0.8185185185185185""","""0.0""","""[0.0, 0.0]""","""0.7425376173235151""","""0.0""","""[0.0, 0.0]""","""True""","""True""","""false"""
"""dense__huydang-dek21-embedding…","""dense_only""","""0.837037037037037""","""0.0""","""[0.0, 0.0]""","""0.7163617882034018""","""0.0""","""[0.0, 0.0]""","""True""","""True""","""false"""
"""dense__e5-base-768""","""dense_only""","""0.8407407407407408""","""0.0""","""[0.0, 0.0]""","""0.7060878464719689""","""0.0""","""[0.0, 0.0]""","""True""","""True""","""false"""
"""bm25-only""","""sparse_only""","""0.7888888888888889""","""""","""[, ]""","""0.6477935262641025""","""""","""[, ]""","""True""","""""","""false"""
"""dense-bm25-rescore__e5-small-3…","""dense_bm25_rescore""","""0.862962962962963""","""0.04444444444444451""","""[0.0, 0.10370370370370371]""","""0.7545162744875943""","""0.011978657164079198""","""[-0.027552100572900976, 0.0509…","""False""","""False""","""false"""
…,…,…,…,…,…,…,…,…,…,…
"""hybrid-tfidf-weighted__e5-smal…","""hybrid_dense_sparse""","""0.8407407407407408""","""0.022222222222222254""","""[-0.0037037037037037034, 0.059…","""0.7046939953449163""","""-0.03784362197859881""","""[-0.08511264158946541, 0.00679…","""False""","""False""","""false"""
"""hybrid-tfidf-rrf__huydang-dek2…","""hybrid_dense_sparse""","""0.8777777777777778""","""0.040740740740740744""","""[-0.040740740740740744, 0.1296…","""0.7322407799485564""","""0.015878991745154614""","""[-0.052686399080960906, 0.0854…","""False""","""False""","""false"""
"""hybrid-tfidf-weighted__huydang…","""hybrid_dense_sparse""","""0.9111111111111111""","""0.07407407407407407""","""[0.018518518518518517, 0.14814…","""0.7424193582848888""","""0.026057570081487014""","""[-0.02903518704027532, 0.08196…","""False""","""False""","""false"""


## 7. Kết luận Thực nghiệm

- **Hiệu năng Hybrid Fusion**: Các phương pháp kết hợp Hybrid (đặc biệt là Min-Max Weighted Fusion) mang lại sự cải thiện rõ rệt về độ phủ Recall@5 so với các mô hình Dense thuần túy trên toàn bộ tập câu hỏi.
- **Category Guardrails & Tính Toàn vẹn**: Đánh giá nghiêm ngặt theo từng phân loại câu hỏi (Category Guardrails) cho thấy một số biến thể fusion có sự xáo trộn nhỏ trong thứ hạng nội bộ của danh mục `relationship` ($n = 14$), do đó hệ thống fail-closed không chọn finalist tự động khi chưa vượt qua đầy đủ các ràng buộc khoa học.
- **Không Cutover Production**: Toàn bộ kết quả và artifact phục vụ cho việc đối soát và báo cáo thực nghiệm, không tác động đến runtime hay collection production đang phục vụ.